In [1]:
import pandas as pd
import numpy as np
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data = load_all_data()
cf   = data['cashflow']
pl   = data['profitandloss']

# Merge cash flow with P&L for combined KPIs
merged = pd.merge(
    cf,
    pl[['company_id', 'year', 'sales', 'net_profit', 'operating_profit']],
    on=['company_id', 'year'],
    how='inner'
)

print(f"Merged rows: {len(merged)}")
print(f"Columns: {merged.columns.tolist()}")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Merged rows: 1145
Columns: ['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow', 'sales', 'net_profit', 'operating_profit']


In [2]:
def compute_fcf(df):
    """
    Free Cash Flow = CFO + CFI
    Positive = company generating cash after investments
    Negative = company burning cash
    """
    df = df.copy()
    df['free_cash_flow_cr'] = (
        df['operating_activity'] + df['investing_activity']
    ).round(2)
    return df

merged = compute_fcf(merged)

# Show FCF for latest year
latest = merged[merged['year'] == '2024-03']
print("Top 5 FCF Generators (FY2024):")
print(latest.nlargest(5, 'free_cash_flow_cr')[
    ['company_id', 'operating_activity',
     'investing_activity', 'free_cash_flow_cr']
].to_string(index=False))

print("\nBottom 5 FCF (Highest Cash Burn):")
print(latest.nsmallest(5, 'free_cash_flow_cr')[
    ['company_id', 'free_cash_flow_cr']
].to_string(index=False))

Top 5 FCF Generators (FY2024):
company_id  operating_activity  investing_activity  free_cash_flow_cr
       TCS             44338.0              6091.0            50429.0
  RELIANCE            158788.0           -113581.0            45207.0
TATAMOTORS             67915.0            -22782.0            45133.0
      ONGC             99263.0            -57203.0            42060.0
       IOC             71099.0            -31464.0            39635.0

Bottom 5 FCF (Highest Cash Burn):
company_id  free_cash_flow_cr
       PFC          -101229.0
BAJFINANCE           -79931.0
BAJAJFINSV           -79634.0
    RECLTD           -59554.0
  CHOLAFIN           -38538.0


In [3]:
def compute_cfo_quality(df):
    """
    CFO Quality = CFO / Net Profit
    > 1.0 = High quality earnings (cash > profit)
    < 0.5 = Accrual risk (profit not backed by cash)
    None  if net profit is zero
    """
    df = df.copy()
    df['cfo_quality_score'] = np.where(
        df['net_profit'] != 0,
        (df['operating_activity'] / df['net_profit']).round(2),
        np.nan
    )
    df['cfo_quality_score'] = pd.to_numeric(
        df['cfo_quality_score'], errors='coerce'
    )

    # Quality label
    def quality_label(score):
        if pd.isna(score):
            return 'N/A'
        elif score > 1.0:
            return 'High Quality'
        elif score > 0.5:
            return 'Moderate'
        else:
            return 'Accrual Risk'

    df['cfo_quality_label'] = df['cfo_quality_score'].apply(quality_label)
    return df

merged = compute_cfo_quality(merged)

print("CFO Quality Distribution (FY2024):")
latest = merged[merged['year'] == '2024-03']
print(latest['cfo_quality_label'].value_counts().to_string())

print("\nSample High Quality companies:")
hq = latest[latest['cfo_quality_label'] == 'High Quality']
print(hq[['company_id', 'cfo_quality_score']].head(5).to_string(index=False))

CFO Quality Distribution (FY2024):
cfo_quality_label
High Quality    56
Accrual Risk    24
Moderate        17

Sample High Quality companies:
company_id  cfo_quality_score
       ABB               5.03
ADANIENSOL               5.05
  ADANIENT               3.09
ADANIGREEN               6.12
ADANIPORTS               1.85


In [4]:
def compute_capex_intensity(df):
    """
    CapEx Intensity = |Investing Activity| / Sales × 100
    < 3%  = Asset light (IT, FMCG)
    > 8%  = Capital intensive (Steel, Power)
    None  if sales is zero
    """
    df = df.copy()
    df['capex_cr'] = df['investing_activity'].abs().round(2)

    df['capex_intensity_pct'] = np.where(
        df['sales'] > 0,
        (df['capex_cr'] / df['sales'] * 100).round(2),
        np.nan
    )
    df['capex_intensity_pct'] = pd.to_numeric(
        df['capex_intensity_pct'], errors='coerce'
    )

    def capex_label(pct):
        if pd.isna(pct):
            return 'N/A'
        elif pct < 3:
            return 'Asset Light'
        elif pct < 8:
            return 'Moderate'
        else:
            return 'Capital Intensive'

    df['capex_label'] = df['capex_intensity_pct'].apply(capex_label)
    return df

merged = compute_capex_intensity(merged)

print("CapEx Intensity Distribution (FY2024):")
latest = merged[merged['year'] == '2024-03']
print(latest['capex_label'].value_counts().to_string())

print("\nMost Capital Intensive companies:")
print(latest.nlargest(5, 'capex_intensity_pct')[
    ['company_id', 'capex_intensity_pct', 'capex_label']
].to_string(index=False))

CapEx Intensity Distribution (FY2024):
capex_label
Capital Intensive    48
Moderate             25
Asset Light          24

Most Capital Intensive companies:
company_id  capex_intensity_pct       capex_label
ADANIGREEN               228.42 Capital Intensive
 ICICIBANK                90.74 Capital Intensive
       ABB                84.51 Capital Intensive
    JIOFIN                77.68 Capital Intensive
 JSWENERGY                71.37 Capital Intensive


In [5]:
def get_capital_allocation_pattern(cfo, cfi, cff):
    """
    Classifies company capital allocation based on
    sign pattern of CFO, CFI and CFF.

    Returns descriptive label for each pattern.
    """
    cfo_sign = '+' if cfo >= 0 else '-'
    cfi_sign = '+' if cfi >= 0 else '-'
    cff_sign = '+' if cff >= 0 else '-'
    pattern  = f"({cfo_sign},{cfi_sign},{cff_sign})"

    pattern_map = {
        '(+,-,-)': 'Reinvestor',
        '(+,-,+)': 'Aggressive Expander',
        '(+,+,-)': 'Asset Seller',
        '(+,+,+)': 'Cash Accumulator',
        '(-,-,+)': 'Distress Signal',
        '(-,+,+)': 'Distress Signal',
        '(-,-,-)': 'Contraction',
        '(-,+,-)': 'Restructuring',
    }

    return pattern, pattern_map.get(pattern, 'Other')

# Apply to all rows
merged[['cf_pattern', 'cf_label']] = merged.apply(
    lambda row: pd.Series(get_capital_allocation_pattern(
        row['operating_activity'],
        row['investing_activity'],
        row['financing_activity']
    )), axis=1
)

print("Capital Allocation Patterns (FY2024):")
latest = merged[merged['year'] == '2024-03']
print(latest['cf_label'].value_counts().to_string())

print("\nDistress Signal companies:")
distress = latest[latest['cf_label'] == 'Distress Signal']
if len(distress) > 0:
    print(distress[['company_id', 'cf_pattern', 'cf_label']].to_string(index=False))
else:
    print("  None! ✅")

Capital Allocation Patterns (FY2024):
cf_label
Reinvestor             59
Aggressive Expander    13
Distress Signal        13
Asset Seller            9
Restructuring           2
Contraction             1

Distress Signal companies:
company_id cf_pattern        cf_label
  AXISBANK    (-,-,+) Distress Signal
BAJAJFINSV    (-,-,+) Distress Signal
BAJFINANCE    (-,-,+) Distress Signal
BANKBARODA    (-,-,+) Distress Signal
      BHEL    (-,+,+) Distress Signal
  CHOLAFIN    (-,-,+) Distress Signal
    GRASIM    (-,-,+) Distress Signal
       M&M    (-,-,+) Distress Signal
       PFC    (-,-,+) Distress Signal
       PNB    (-,-,+) Distress Signal
    RECLTD    (-,-,+) Distress Signal
SHRIRAMFIN    (-,-,+) Distress Signal
  TVSMOTOR    (-,-,+) Distress Signal


In [6]:
print("Cash Flow KPIs — Coverage Summary:")
print(f"  Total rows:              {len(merged)}")
print(f"  FCF computed:            {merged['free_cash_flow_cr'].notna().sum()}")
print(f"  CFO Quality computed:    {merged['cfo_quality_score'].notna().sum()}")
print(f"  CapEx Intensity:         {merged['capex_intensity_pct'].notna().sum()}")
print(f"  Capital Allocation:      {merged['cf_label'].notna().sum()}")

print("\nSample — TCS latest year:")
tcs = merged[merged['company_id'] == 'TCS'].tail(1)[[
    'company_id', 'year', 'free_cash_flow_cr',
    'cfo_quality_score', 'cfo_quality_label',
    'capex_intensity_pct', 'cf_label'
]]
print(tcs.to_string(index=False))

print("\nSample — RELIANCE latest year:")
rel = merged[merged['company_id'] == 'RELIANCE'].tail(1)[[
    'company_id', 'year', 'free_cash_flow_cr',
    'cfo_quality_score', 'cfo_quality_label',
    'capex_intensity_pct', 'cf_label'
]]
print(rel.to_string(index=False))

Cash Flow KPIs — Coverage Summary:
  Total rows:              1145
  FCF computed:            1143
  CFO Quality computed:    1142
  CapEx Intensity:         1142
  Capital Allocation:      1145

Sample — TCS latest year:
company_id    year  free_cash_flow_cr  cfo_quality_score cfo_quality_label  capex_intensity_pct     cf_label
       TCS 2024-03            50429.0               0.96          Moderate                 2.53 Asset Seller

Sample — RELIANCE latest year:
company_id    year  free_cash_flow_cr  cfo_quality_score cfo_quality_label  capex_intensity_pct   cf_label
  RELIANCE 2024-03            45207.0               2.01      High Quality                12.63 Reinvestor
